# The Architecture of a Layover: reproducible pilot
Source: Dillon Wong, Flight Prices (2022), CC BY 4.0. See DATA_README.md.
Run from this directory with Python 3. Outputs go to reproduced/. Standard library only.
This is a convenience extract of advertised schedules, not a passenger sample. Nearby endpoint exclusions are a scope filter.

In [1]:
import csv,json,hashlib,collections,statistics,datetime,pathlib,shutil,math
base=pathlib.Path.cwd(); p=base/'reproduced'; p.mkdir(parents=True,exist_ok=True)
shutil.copyfile(base/'source_bos_lax.csv',p/'source_bos_lax.csv')
rows=list(csv.DictReader(open(p/'source_bos_lax.csv',encoding='utf-8')))
keep={}; excluded=collections.Counter(); errors=[]
for rownum,r in enumerate(rows,2):
 arr=r['segmentsArrivalTimeEpochSeconds'].split('||'); dep=r['segmentsDepartureTimeEpochSeconds'].split('||')
 if len(arr)!=2: excluded['nonstop' if len(arr)==1 else 'multiple_connections']+=1; continue
 fields=['segmentsDepartureTimeEpochSeconds','segmentsArrivalTimeEpochSeconds','segmentsDepartureTimeRaw','segmentsArrivalTimeRaw','segmentsDepartureAirportCode','segmentsArrivalAirportCode','segmentsAirlineCode','segmentsCabinCode','segmentsDurationInSeconds']
 try:
  a={k:r[k].split('||') for k in fields}; assert all(len(v)==2 for v in a.values()), 'segment_field_length'
  assert a['segmentsArrivalAirportCode'][0]==a['segmentsDepartureAirportCode'][1], 'airport_change_connection'
  assert a['segmentsDepartureAirportCode'][0]==r['startingAirport'] and a['segmentsArrivalAirportCode'][1]==r['destinationAirport'], 'nearby_airport_endpoint'
  arr=list(map(int,arr)); dep=list(map(int,dep)); assert all(dep[i]<arr[i] for i in range(2)); assert dep[1]>arr[0]
  for key,epoch in [('segmentsArrivalTimeRaw',arr),('segmentsDepartureTimeRaw',dep)]:
   assert all(int(datetime.datetime.fromisoformat(a[key][i]).timestamp())==epoch[i] for i in range(2))
  assert all(arr[i]-dep[i]==int(a['segmentsDurationInSeconds'][i]) for i in range(2))
  t0=datetime.datetime.fromisoformat(a['segmentsArrivalTimeRaw'][0]); t1=datetime.datetime.fromisoformat(a['segmentsDepartureTimeRaw'][1]); assert t0.utcoffset()==t1.utcoffset()
  night=0; day=t0.date()-datetime.timedelta(days=1)
  while day<=t1.date():
   n0=datetime.datetime.combine(day,datetime.time(22),tzinfo=t0.tzinfo); n1=n0+datetime.timedelta(hours=8)
   night+=max(0,(min(t1,n1)-max(t0,n0)).total_seconds())/60; day+=datetime.timedelta(days=1)
 except Exception as e: errors.append({'row':rownum,'error':str(e)}); continue
 sig='|'.join(r[k] for k in ['flightDate','startingAirport','destinationAirport']+fields[:2]+fields[4:8]); sid=hashlib.sha256(sig.encode()).hexdigest()[:16]
 if sid in keep: keep[sid]['source_offer_count']+=1; keep[sid]['source_csv_rows']+=';'+str(rownum); excluded['duplicate_schedule_offers']+=1; continue
 keep[sid]={'itinerary_id':sid,'search_date':r['searchDate'],'flight_date':r['flightDate'],'origin':r['startingAirport'],'destination':r['destinationAirport'],'direction':r['startingAirport']+'-'+r['destinationAirport'],'hub':a['segmentsArrivalAirportCode'][0],'inbound_carrier':a['segmentsAirlineCode'][0],'outbound_carrier':a['segmentsAirlineCode'][1],'carrier_pair':' / '.join(a['segmentsAirlineCode']),'cabin_pair':' / '.join(a['segmentsCabinCode']),'hub_arrival_local':t0.isoformat(),'hub_departure_local':t1.isoformat(),'hub_arrival_epoch':arr[0],'hub_departure_epoch':dep[1],'layover_minutes':(dep[1]-arr[0])/60,'night_overlap_minutes':night,'crosses_local_midnight':int(t1.date()>t0.date()),'long_layover_6h':int(dep[1]-arr[0]>=21600),'source_offer_count':1,'source_csv_rows':str(rownum)}
clean=list(keep.values())
def write(name,data):
 with (p/name).open('w',newline='',encoding='utf-8') as f:
  w=csv.DictWriter(f,fieldnames=list(data[0])); w.writeheader(); w.writerows(data)
write('layovers_clean.csv',clean)
def pct(a,q):
 a=sorted(a); pos=(len(a)-1)*q; lo=int(pos); hi=math.ceil(pos); return a[lo]+(a[hi]-a[lo])*(pos-lo)
summ=[]
for direction in ['All','BOS-LAX','LAX-BOS']:
 groups=collections.defaultdict(list)
 for r in clean:
  if direction=='All' or r['direction']==direction: groups[r['hub']].append(r)
 for hub,rr in groups.items():
  vals=[r['layover_minutes'] for r in rr]; summ.append({'direction':direction,'hub':hub,'n':len(rr),'mean_minutes':statistics.mean(vals),'median_minutes':statistics.median(vals),'p90_minutes':pct(vals,.9),'min_minutes':min(vals),'max_minutes':max(vals),'midnight_count':sum(r['crosses_local_midnight'] for r in rr),'long_6h_count':sum(r['long_layover_6h'] for r in rr),'night_overlap_count':sum(r['night_overlap_minutes']>0 for r in rr),'flight_dates':len(set(r['flight_date'] for r in rr))})
write('hub_summary.csv',sorted(summ,key=lambda r:(r['direction'],-r['n'])))
meta={'raw_rows':len(rows),'raw_one_stop':len(rows)-excluded['nonstop']-excluded['multiple_connections'],'excluded':dict(excluded),'validation_exclusion_counts':dict(collections.Counter(e['error'] for e in errors)),'validation_errors':errors,'unique_schedules':len(clean),'flight_date_min':min(r['flight_date'] for r in clean),'flight_date_max':max(r['flight_date'] for r in clean),'hubs':len(set(r['hub'] for r in clean)),'overall_mean':statistics.mean(r['layover_minutes'] for r in clean),'overall_median':statistics.median(r['layover_minutes'] for r in clean),'midnight_count':sum(r['crosses_local_midnight'] for r in clean),'long_6h_count':sum(r['long_layover_6h'] for r in clean),'min_layover':min(r['layover_minutes'] for r in clean),'max_layover':max(r['layover_minutes'] for r in clean),'raw_sha256':hashlib.sha256((p/'source_bos_lax.csv').read_bytes()).hexdigest()}
(p/'quality_checks.json').write_text(json.dumps(meta,indent=2)); shutil.copyfile(base/'extraction.json',p/'extraction.json')
print({k:v for k,v in meta.items() if k != 'validation_errors'})


{'raw_rows': 1392, 'raw_one_stop': 1005, 'excluded': {'nonstop': 372, 'multiple_connections': 15}, 'validation_exclusion_counts': {'nearby_airport_endpoint': 177}, 'unique_schedules': 828, 'flight_date_min': '2022-04-17', 'flight_date_max': '2022-04-26', 'hubs': 26, 'overall_mean': 137.92753623188406, 'overall_median': 96.0, 'midnight_count': 21, 'long_6h_count': 54, 'min_layover': 31.0, 'max_layover': 551.0, 'raw_sha256': 'c56acae5318727ba54b43bb70d548820fb6bed8d4974be1630ed3bd464427ed9'}


## Independent verification
Recompute waits from local timestamps, compare with the epoch-based output, and reconcile the published claims.

In [2]:
published=list(csv.DictReader(open(base/'layovers_clean.csv',encoding='utf-8')))
assert len(published)==len(clean)==828
for old,new in zip(published,clean):
    assert old['itinerary_id']==new['itinerary_id']
    assert float(old['layover_minutes'])==new['layover_minutes']
independent=[]
for r in rows:
    aa=r['segmentsArrivalAirportCode'].split('||'); dd=r['segmentsDepartureAirportCode'].split('||')
    if len(aa)!=2 or dd[0]!=r['startingAirport'] or aa[-1]!=r['destinationAirport']: continue
    arrive=datetime.datetime.fromisoformat(r['segmentsArrivalTimeRaw'].split('||')[0])
    depart=datetime.datetime.fromisoformat(r['segmentsDepartureTimeRaw'].split('||')[1])
    independent.append((depart-arrive).total_seconds()/60)
assert len(independent)==828
assert abs(statistics.mean(independent)-meta['overall_mean'])<1e-9
assert statistics.median(independent)==96
assert sum(x>=360 for x in independent)==54
assert 1392==372+15+177+828
assert len(set(r['itinerary_id'] for r in clean))==828
assert sum(r['crosses_local_midnight'] for r in clean)==21
assert sum(r['night_overlap_minutes']>0 for r in clean)==144
print('PASS: all 828 durations reproduced; mean, median, long waits, midnight crossings, and row counts reconcile.')


PASS: all 828 durations reproduced; mean, median, long waits, midnight crossings, and row counts reconcile.
